# L42 - Deep RL Training Starter

**Learning objectives**
- Train a DQN agent on a small queue-control environment.
- Distinguish the environment API from the learning algorithm API.
- Evaluate a trained policy against simple heuristics.
- Identify what must change before moving from a toy environment to a full clinic wrapper.

In [ ]:
import gymnasium as gym
import numpy as np
import simpy

from simdes.envs.base_env import SimPyEnv

try:
    from stable_baselines3 import DQN
    SB3_AVAILABLE = True
except ImportError:
    SB3_AVAILABLE = False
    print('Install simdes[rl] or stable-baselines3 + torch to run the training cells.')

In [ ]:
class ToySimPyQueueEnv(SimPyEnv):
    def __init__(self, sim_time=40.0, decision_interval=2.0, seed=None):
        super().__init__(sim_time=sim_time, seed=seed)
        self.decision_interval = decision_interval
        self.observation_space = gym.spaces.Box(
            low=np.array([0.0, 0.0], dtype=np.float32),
            high=np.array([20.0, 1.0], dtype=np.float32),
            dtype=np.float32,
        )
        self.action_space = gym.spaces.Discrete(2)

    def _build_sim(self):
        self.queue = 0
        self.capacity = 1
        self._env.process(self._arrival_process())
        self._env.process(self._service_process())

    def _arrival_process(self):
        while True:
            yield self._env.timeout(1.0)
            self.queue = min(20, self.queue + int(self._rng.poisson(1.25)))

    def _service_process(self):
        while True:
            yield self._env.timeout(1.0)
            self.queue = max(0, self.queue - self.capacity)

    def _get_obs(self):
        return np.array([self.queue, min(self._env.now / self.sim_time, 1.0)], dtype=np.float32)

    def _step_sim(self, action):
        self.capacity = 1 + int(action)
        target = min(self.sim_time, self._env.now + self.decision_interval)
        self._env.run(until=target)
        return -(self.queue + 0.25 * int(action))

In [ ]:
def evaluate_policy(policy_fn, n_episodes=20):
    returns = []
    for seed in range(n_episodes):
        env = ToySimPyQueueEnv(seed=seed)
        obs, _ = env.reset(seed=seed)
        done = False
        total = 0.0
        while not done:
            action = int(policy_fn(obs))
            obs, reward, terminated, truncated, _ = env.step(action)
            total += reward
            done = terminated or truncated
        returns.append(total)
    return np.array(returns)

if SB3_AVAILABLE:
    train_env = ToySimPyQueueEnv(seed=0)
    model = DQN('MlpPolicy', train_env, learning_rate=1e-3, exploration_fraction=0.4, verbose=0)
    model.learn(total_timesteps=4000)

    dqn_returns = evaluate_policy(lambda obs: model.predict(obs, deterministic=True)[0])
    hold_returns = evaluate_policy(lambda obs: 0)
    extra_returns = evaluate_policy(lambda obs: 1)

    print('DQN mean return   :', float(np.mean(dqn_returns)))
    print('Hold mean return  :', float(np.mean(hold_returns)))
    print('Extra mean return :', float(np.mean(extra_returns)))
else:
    print('Training skipped because stable-baselines3 is not installed.')

## Try It Yourself

1. Increase the arrival rate and compare DQN against the two heuristic baselines again.
2. Swap DQN for PPO if your environment includes the optional RL dependencies.
3. Replace the toy environment with a fully implemented clinic wrapper once the package-level `ClinicEnv` is ready.